# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型  |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** どこから読み、最初に何をするか |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 新しいホストを作る・移す

**VPS を作り直す・別の業者へ移すときの流れ。** 一度きりの作業なので、細部は以前の手順書([../Old/docs/](../Old/docs/))にある。
ここは**順番と、そこから叩くもの**をまとめる。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

> **LAN の検証機(ローカル環境)を立てるときは [13-local-env](13-local-env.ipynb)。** ここの手順は公開 DNS と Let's Encrypt が前提で、LAN では証明書が取れずに再起動を繰り返す。

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 0. 全体の順番

| # | やること | 詳しい手順 |
|---|---|---|
| 1 | VPS の初期設定(作業用ユーザー `km`・SSH を締める・ファイアウォール・Docker) | [vps-setup.md](../Old/docs/vps-setup.md) |
| 1-2 | **配備用の `kmops` を作る**(docker あり・sudo なし)。最初の配備だけ `km` で行い、そのあと `scripts/host-ops-user.sh create` → `handover` で置き場と門番を渡す | [12](12-hardening-2026-09-15.ipynb) §7-4 B |
| 2 | 業者のパケットフィルタで 22/80/443 以外を落とす。**ufw は Docker が publish したポートを塞げない** | vps-setup.md §4 |
| 3 | DNS(A / MX 不要 / SPF / DKIM / DMARC)と送信専用メールサーバー | [mail-server.md](../Old/docs/mail-server.md) |
| 4 | 旧ホストで控えを取る(**入口を閉じてから**) | [03-backup](03-backup.ipynb) |
| 5 | 新ホストに `.env` を置く(**URL 系で要るのは `KM_DOMAIN` だけ**。**`POSTGRES_USER` は名前まで旧ホストと同じ**) | [vps-migration.md](../Old/docs/vps-migration.md) §3 / 下の §2 |
| 6 | コードを配備して起動、`composer install` | [02-deploy](02-deploy.ipynb) |
| 7 | データを復元、`config/*.local.php` を置く | [03-backup](03-backup.ipynb) §8 / vps-migration.md §5 |
| 8 | 証明書を取る(**まず dry-run**。発行回数を消費しない) | vps-migration.md §6 |
| 9 | Logto の向き先(リソース・リダイレクト URI)と SMTP コネクター・テンプレート9種の URL | [data-migration.md](../Old/docs/data-migration.md) §6 / [logto-smtp.md](../Old/docs/logto-smtp.md) |
| 10 | 仕上げ(イベントモードの表・HSTS は**公的な証明書になってから**)、定期処理の仕込み | vps-migration.md §8 / [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) |
| 11 | 通しで点検 | [vps-checklist.md](../Old/docs/vps-checklist.md) の A・B |
| 12 | DNS を切り替える。**旧ホストはすぐ止めない**(数時間〜1日残す)。切り替えたら旧ホストの入口は止める | vps-migration.md §10 |

**ドメインも変わる移行**なら、切り替えのあとで §6 の 6〜8(Logto の戻り先・メール・Android)も通す。

**当日やらないこと**も vps-migration.md の末尾にある。フロント資材の入手元は [vendor-assets.md](../Old/docs/vendor-assets.md)。

## 1. VPS の初期設定(Old/setup-vps.ps1)

**まず `-DryRun`**(リモートは一切変更しない)。`203.0.113.10` を新しい VPS の IP に書き換える。
SSH を締める(`-HardenSsh`)のは、作業用ユーザーで入れることを確かめてから。

### 何をするかだけ見る

新しいホストに繋ぐが、変更はしない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
..\Old\setup-vps.ps1 -HostName 203.0.113.10 -AcceptHostKey -DryRun

### 初期設定する

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "新しい VPS を初期設定します(SSH も締めます)"
..\Old\setup-vps.ps1 -HostName 203.0.113.10 -HardenSsh

## 2. 新しいホストへ配備する

`-HostName` などを直接渡す。初めて繋ぐので `-AcceptHostKey`。

### `.env` の URL 系は `KM_DOMAIN` だけでよい(2026-09-14 から)

`compose.vps.yaml` が `KM_DOMAIN` から URL 系を導く。**秘密の値(DB のパスワード・Logto の資格情報など)は従来どおり要る。**

| 導かれる値 | 既定 |
|---|---|
| `KM_APP_URL` / `APP_URL` | `https://<KM_DOMAIN>` |
| `KM_APP_PORT` | `443` |
| `KM_CERT` / `KM_CERT_KEY` | `/etc/letsencrypt/live/<KM_DOMAIN>/fullchain.pem`・`privkey.pem` |
| `LOGTO_ENDPOINT` / `LOGTO_ADMIN_ENDPOINT` | `https://<KM_DOMAIN>:3001` / `:3002` |
| `PMA_ABSOLUTE_URI` | `https://<KM_DOMAIN>:8281/` |
| `MAIL_HOST` / `MAIL_PORT` / `MAIL_FROM` | `mailserver` / `587` / `noreply@<KM_DOMAIN>` |
| mailserver の `ALLOWED_SENDER_DOMAINS` / `POSTFIX_myhostname` | `<KM_DOMAIN>` / `mail.<KM_DOMAIN>` |
| `KM_API_RESOURCE`(web の `LOGTO_API_RESOURCE`・`KOSENMAP_LOGTO_AUDIENCE`) | `https://<KM_DOMAIN>/api` —— **ドメインが変わる移行では旧の値を書く**(§6) |

- **`.env` に書けばそちらが勝つ**(空の行は書いていないのと同じ)。いまの本番の `.env` は URL 系が全部明示で入った形で、そのままでも動く
- **`KM_DOMAIN` が無いと `docker compose config` の時点で止まる**(以前の既定 `ito8795.com` へ黙って落ちない)
- `COMPOSE_FILE="compose.yaml:compose.vps.yaml"` は要る(導出は VPS の構成にしかない)
- 置いたら §6 の「6-0. いま compose が読んでいる値」で確かめられる(新しいホストに向けるなら、読み込む前に `KM_HOST` を置く)

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%ps --confirm "新しいホストへ配備して up -d します"
.\deploy-to-host.ps1 -HostName 203.0.113.10 -User kmops -KeyPath "$env:USERPROFILE\.ssh\km_ops" -RemotePath /opt/kosenmap -AcceptHostKey -Action up -Yes

## 3. 復元

[03-backup](03-backup.ipynb) の §8。**移行先で Logto を一度起動してから**戻す(`logto_tenant_logto` ロールが要る。無ければスクリプトが何も変えずに止まる)。

## 4. Logto の向き先を変える

DB を移しただけでは、サインインが一歩も進まない。**エラーの文言で切り分ける。**

| 症状 | 原因 | 直すもの |
|---|---|---|
| `invalid_client` | `.env` の `LOGTO_APP_ID` が復元した Logto に無い | `.env` を**復元した Logto に合わせる**(新しく登録しない) |
| `invalid_target` | API リソースが旧ホストの URL のまま | `resources.indicator` |
| コールバックで弾かれる | リダイレクト URI が旧ホストのまま | アプリの `redirectUris` / `postLogoutRedirectUris` |

Console(3002)で直すのが本筋。SQL での直し方と確かめ方は [data-migration.md](../Old/docs/data-migration.md) §6。**変えたら Logto を再起動する。**

**ドメインを変えるときは API リソース(`resources.indicator`)を変えない**(§6)。`invalid_target` が出たら、まず `.env` の `KM_API_RESOURCE`(空なら `https://<KM_DOMAIN>/api`)が Logto の indicator と同じかを見る。
リダイレクト URI などは §6 の `logto-domain.php` でまとめて書き換えられる。

## 5. 切り替えたあと

- [ ] [01-daily-check](01-daily-check.ipynb) を上から流す
- [ ] [05-containers](05-containers.ipynb) の §7(画面での確認)
- [ ] 定期処理を仕込み([04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) §3)、試しのメールが届く
- [ ] この PC の週次タスクの `-HostName` を新しいホストにする([03-backup](03-backup.ipynb) §5)
- [ ] 旧ホストの入口を止める。**両方に書き込みが起きると、機械的には合成できない**

## 6. ドメインを変える

**同じホストのまま、名前だけを移す**ときの順番(`ito4.jp` → 別のドメイン)。止まるのは 6-5 の立ち上げ直しの数十秒だけ。
**サーバー側で書き換えるのは `.env` の `KM_DOMAIN` 1行だけ**で、URL 系は `compose.vps.yaml` が導く(§2)。スクリプトは `.env` を書き換えるところまでで、**立ち上げ直しは人が行う。**

| # | やること | 印 | 叩くもの |
|---|---|---|---|
| 6-1 | DNS の A レコード(新しい名前と `mail.<新>`)をこのホストへ | 手作業 / 🟢 | 業者の DNS の画面 |
| 6-2 | 新しい名前の証明書を取る(先に試しの発行) | 🔴 | `scripts/host-cert.sh issue` |
| 6-3 | 何が変わるかを見る | 🟢 | `scripts/host-domain.sh check` |
| 6-4 | `.env` を控えてから書き換える | 🔴 | `scripts/host-domain.sh apply` |
| 6-5 | 立ち上げ直す・証明書と応答を見る | 🔴 / 🟢 | `docker compose up -d` / `host-cert.sh status` |
| 6-6 | Logto の戻り先の一覧 | 🟢 | `scripts/logto-domain.php --from=<旧>` |
| 6-7 | Logto の戻り先を書き換える | 🔴 | 同じコマンドに `--apply` |
| 6-8 | メール(DKIM・SPF・DMARC・PTR)・reCAPTCHA・Android の再ビルド・旧の証明書 | 手作業 / 🔴 | 下の 6-8 |

**Android は 6-5 の瞬間から繋がらなくなる**(配布済みのアプリは旧の名前へ繋ぎに行き、nginx は新しい名前の証明書しか出さない)。新しい APK は先に作っておく(6-8)。

### 変えないもの: `KM_API_RESOURCE`

アクセストークンの audience(Logto に登録した API リソースの識別子)。**URL の形をした名前にすぎず、その名前で接続するわけではない。**
ドメインから導いたままだと、`KM_DOMAIN` を変えた瞬間に web が受け付ける audience も変わり、**配布済みのアプリのトークンが全部通らなくなる**
(アプリは旧の識別子でトークンを取り、Logto の indicator も変わらない)。
だから `host-domain.sh apply` は、`.env` に行が無ければ**いま使っている値**(動いている web の `KOSENMAP_LOGTO_AUDIENCE`。取れなければ `https://<旧>/api`)を `KM_API_RESOURCE` として書き残す。
**Logto の API リソースも、Android の `kosenmap.apiResource` も据え置く。** `logto-domain.php` も API リソースには触れない。

### 書き換える前に止まるもの(`host-domain.sh`)

- 新しい名前の証明書が無い —— 書き換えてから気づくと次の `up` で nginx が起動できず、**HSTS のため誰もサイトに入れない**
- `compose.vps.yaml` が `KM_DOMAIN` から導く版でない(先に配備する)—— コメントにした行が校内 LAN の既定(192.168.3.29)へ落ちる
- `COMPOSE_FILE` に `compose.vps.yaml` が無い

検査は全部「書き換えた `.env` の写し」で行い、**元の `.env` は最後の1回しか触らない。** 書き換えたあと compose が読めなければ、控えから自動で戻す。

> 下のセルの `OLD` / `NEW` / `EMAIL` は、**自分の値に書き換えてから**実行する(🔴 のセルは例の `example.jp` のままだと止まる)。
> 名前を変えたあとは、この取扱説明書の接続先も変わる —— 読み込む前に環境変数 `KM_HOST` を新しい名前にする([00-start](00-start.ipynb) §5)。

### 6-0. いま compose が読んでいる値

`.env` の `KM_DOMAIN` と `KM_API_RESOURCE`、compose が組み立てた URL 系の値だけを出す(秘密の値は出さない)。始める前と、6-5 のあとに見比べる。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
grep -E '^(COMPOSE_FILE|KM_DOMAIN|KM_API_RESOURCE)=' .env
echo
if ! _cfg="$(docker compose config 2>/dev/null)"; then
  echo "★ docker compose が .env を読めません(エラー文には秘密の断片が出るので表示しません)"
else
  printf '%s\n' "$_cfg" \
    | grep -E '^[[:space:]]+(KM_DOMAIN|KM_APP_URL|KM_CERT|KM_CERT_KEY|APP_URL|ENDPOINT|ADMIN_ENDPOINT|LOGTO_ENDPOINT|LOGTO_ADMIN_ENDPOINT|PMA_ABSOLUTE_URI|MAIL_HOST|MAIL_PORT|MAIL_FROM|ALLOWED_SENDER_DOMAINS|POSTFIX_myhostname|LOGTO_API_RESOURCE|KOSENMAP_LOGTO_AUDIENCE):' \
    | sed 's/^[[:space:]]*/  /' | sort -u
fi

### 6-1. DNS

業者の DNS の画面で、新しい名前の **A レコード**をこのホストの IP へ向ける。メールも移すので **`mail.<新>` の A** も同じ IP へ(SPF・DKIM・DMARC は 6-8)。`www.<新>` も使うなら同じく。
広まったかを、このホストから名前を引いて見る。**空か、このホストの IP でなければ次へ進まない**(発行を試すと失敗回数の上限を消費する)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
NEW=kosenmap.example.jp
echo "このホスト: $(hostname -I)"
for _n in "$NEW" "mail.$NEW"; do
  echo "$_n → $(getent ahostsv4 "$_n" | awk '{print $1}' | sort -u | tr '\n' ' ')"
done

### 6-2. 証明書を取る

`host-cert.sh issue` は、DNS がこのホストを指しているか → 80 番の `/.well-known/acme-challenge/` が外から届くか を確かめてから発行する。
**まず `--dry-run`(試験用の発行元。発行回数を消費しない)が通ってから本物を取る**(下のセルは続けて流す)。
`EMAIL` は Let's Encrypt から期限切れの警告を受け取る宛先。`www.<新>` も証明書に含めるなら、2行とも `--www` を足す。
**取るだけで、`.env` と nginx は変えない**(サイトはまだ旧の名前のまま)。取れたら letsencrypt の置き場に `live/<新>/` が増える。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "新しいドメインの証明書を Let's Encrypt から取ります(.env と nginx は変えません)" --timeout 900
NEW=kosenmap.example.jp
EMAIL=admin@example.jp
case "$NEW $EMAIL" in
  *example.jp*) echo "★ NEW と EMAIL を書き換えてから実行してください(何もしていません)"; exit 2 ;;
esac
./scripts/host-cert.sh --path /opt/kosenmap issue --domain "$NEW" --email "$EMAIL" --dry-run </dev/null \
  && ./scripts/host-cert.sh --path /opt/kosenmap issue --domain "$NEW" --email "$EMAIL" </dev/null

### 6-3. 何が変わるかを見る(host-domain.sh check)

構成(`COMPOSE_FILE`)→ `.env` のどの行をコメントにするか → 書き換えたあと compose が読む値 → 新しい名前の証明書 → DNS を順に出す。
**★ が出たら止まり、`.env` は変えない。** 「手で確かめる」に出たキー(旧ドメインを含むが導出の対象ではないもの。例: `MAIL_ADMIN_TO`)は、値を自分で見て直す。
`.env` の写しを作ってすぐ消すほかは何も変えない(秘密の値は表示しない)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
NEW=kosenmap.example.jp
./scripts/host-domain.sh --path /opt/kosenmap check "$NEW"

### 6-4. `.env` を書き換える(host-domain.sh apply)

控え `.env.bak-<日時>`(600)を取ってから、`KM_DOMAIN` を新しい名前にし、旧ドメインを含む URL 系の行
(`KM_APP_URL` `KM_CERT` `KM_CERT_KEY` `APP_URL` `LOGTO_ENDPOINT` `LOGTO_ADMIN_ENDPOINT` `PMA_ABSOLUTE_URI` `MAIL_FROM`)を**消さずにコメントにして**導出に任せる。
`KM_API_RESOURCE` の行が無ければ(空の行も)、いまの値を書き足す。check と同じ検査を先に通し、★ があれば何も変えない。
**立ち上げ直しはしない。** 最後に「次にやること」と戻し方が出る(この下の 6-5〜6-8 と同じ)ので、続けて 6-5 へ進む。
「`.env` を書き換えられません」と出たら持ち主の都合で、何も変えずに止まっている。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の .env の KM_DOMAIN を書き換えます(控えを取ります。立ち上げ直しはしません)" --timeout 300
NEW=kosenmap.example.jp
case "$NEW" in
  *example.jp) echo "★ NEW を書き換えてから実行してください(何もしていません)"; exit 2 ;;
esac
./scripts/host-domain.sh --path /opt/kosenmap apply "$NEW"

### 6-5. 立ち上げ直す

**数十秒、サイトが途切れる。** 設定の変わったサービスが新しい名前で作り直される。**ここから配布済みの Android アプリは繋がらない。**

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "docker compose up -d で新しいドメインに切り替えます(数十秒途切れます)" --timeout 900
docker compose up -d
docker compose ps

証明書と応答を見る。新しい名前の行に「nginx が出している証明書と一致しています」と出て、下の2つが `200` ならよい。
旧の名前の証明書の行も残るが、それは 6-8 で消す。6-0 のセルも流して、URL 系が新しい名前になったことを見比べる。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 120
NEW=kosenmap.example.jp
./scripts/host-cert.sh --path /opt/kosenmap status </dev/null
echo
for _u in "https://$NEW/" "https://$NEW:3001/oidc/.well-known/openid-configuration"; do
  echo "$(curl -s -o /dev/null -m 15 -w '%{http_code}' "$_u")  $_u"
done

### 6-6. Logto の戻り先 —— 一覧だけ(logto-domain.php)

`.env` を変えても **Logto の DB に登録した URL は変わらない。** 旧のままだと、サインインが `redirect_uri` の不一致で必ず落ち(管理画面に入れない)、webhook(アカウント削除の後片付け)も黙って届かなくなる。

`logto-domain.php` は、**ホスト名が旧ドメインと完全に一致する** http(s) の URL だけを新しい名前に換える(スキーム・ポート・パスは保つ)。
対象はアプリのリダイレクト URI・サインアウト後の URI・バックチャネル・ログアウトとロゴの URI・CORS の許可 Origin・webhook の送り先・サインイン画面の利用規約などの URL。
`www.<旧>` や端末アプリのカスタムスキームは変えず「手で確かめる」に出す。**API リソース(audience)と Console(admin テナント)の戻り先は扱わない。** メールのテンプレートの中の URL も見ないので、Console で目で確かめる。

- 新しい名前は `APP_URL` のホスト名(6-5 のあとなら新しい名前)。`--to=<新>` で明示もできる
- **`-u www-data` を外さない。** root だと Management API のトークンのキャッシュ(`src/cache/`)の持ち主が root になり、管理画面が使えなくなる(付け忘れたら止まる)
- 一覧はトークンのキャッシュを置くほかは何も変えない。webhook の URL のクエリは伏せて表示する

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 120
OLD=ito4.jp
docker compose exec -T -u www-data web php scripts/logto-domain.php --from="$OLD"

### 6-7. Logto の戻り先を書き換える

一覧を見て、変わるものに納得してから。書き換えたあと**読み直して**、旧ドメインの URL が残っていないかまで見る(残れば ★ と終了コード 1)。
終わったら、**新しい名前で管理画面にサインインし、戻ってこられる**ことを確かめる。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "Logto に登録してある戻り先の URL を新しいドメインに書き換えます" --timeout 300
OLD=ito4.jp
docker compose exec -T -u www-data web php scripts/logto-domain.php --from="$OLD" --apply

### 6-8. 残りの作業

**メール**(届かなくなるので早めに。送信サーバーの設計と DNS の書き方は [mail-server.md](../Old/docs/mail-server.md))

| 何を | どこで | 値 |
|---|---|---|
| `mail.<新>` の A | DNS | このホストの IP(6-1) |
| SPF | DNS(`<新>` の TXT) | `v=spf1 a:mail.<新> -all` など |
| DMARC | DNS(`_dmarc.<新>` の TXT) | 旧ドメインと同じ方針で |
| DKIM | DNS(`mail._domainkey.<新>` の TXT) | 新しい名前の鍵が作られる。公開鍵は下のセル |
| 逆引き(PTR) | **VPS 業者の管理画面** | `mail.<新>`(mailserver が名乗る `POSTFIX_myhostname` と一致させる。合わないと Gmail が迷惑メールにする) |

登録したら [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) §5 で1通送り、届いたか・迷惑メールに入らないかを見る。

**そのほか**

- **reCAPTCHA**: Google の管理画面で、許可するドメインに `<新>` を足す(問い合わせフォームはホスト名も照合する)
- **Android**: 両フレーバー(visitor / admin)を `-Pkosenmap.domain=<新>` で作り直して配る。**`kosenmap.apiResource` はサーバーの `KM_API_RESOURCE` と同じ値**(既定 `https://ito4.jp/api`。違うとトークンが通らない。一緒に移すなら [14](14-domain-ito4.ipynb) の手順で Logto に先に作る)。6-5 より前に作っておく

  ```powershell
  cd C:\Users\itota\Documents\Test
  .\gradlew.bat assembleVisitorRelease assembleAdminRelease -Pkosenmap.domain=<新> -PkosenmapKeystoreProperties=<パス>
  ```

- **この PC の接続先**: 取扱説明書の `KM_HOST`([00-start](00-start.ipynb) §5)、`deploy-to-host.ps1` と週次タスクの `-HostName`([03-backup](03-backup.ipynb) §5)
- **控え** `/opt/kosenmap/.env.bak-<日時>` には秘密が入っている。落ち着いたら消す
- **旧ドメインの証明書**: DNS を手放すなら、毎日 3:47 の更新が失敗してメールが来るので消す(下のセル)

公開鍵を見る(DNS に登録する値)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker compose logs --no-log-prefix mailserver 2>&1 | grep -A6 DKIM | tail -n 20

旧ドメインの証明書を消す。**`KM_DOMAIN` がまだ旧の名前なら消さずに止まる。** 消した証明書は戻せない(取り直しになる)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "旧ドメインの証明書を letsencrypt の置き場から消します(戻せません)" --timeout 300
OLD=ito4.jp
if grep -qx "KM_DOMAIN=$OLD" .env; then
  echo "★ KM_DOMAIN がまだ $OLD です。消しません"
  exit 2
fi
docker compose exec -T certbot certbot delete --cert-name "$OLD" --non-interactive </dev/null
./scripts/host-cert.sh --path /opt/kosenmap status </dev/null

### 戻すとき

`.env` を控えから戻して立ち上げ直す。`BAK` を 6-4 で出た控えの名前にしてから実行する(無ければ一覧を出して止まる)。
Logto の戻り先を書き換えていたら、戻したあとに 6-6 → 6-7 を `OLD` を新しい名前にして流す(`.env` を戻したあとなら `--to` は要らない)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "本番の .env を控えから戻して立ち上げ直します" --timeout 900
BAK=.env.bak-YYYYMMDD-HHMMSS
if [ ! -s "$BAK" ]; then
  echo "★ $BAK がありません。控えの一覧:"
  ls -1 .env.bak-* 2>/dev/null
  exit 2
fi
cat "$BAK" > .env && docker compose up -d